# Step 7: 권한 시스템 — 다단계 게이트

## 학습 목표

- Claude Code의 **checkPermissions() → hooks → classifier → user dialog** 다단계 파이프라인을 이해합니다
- **PermissionMode** (AUTO, ASK, DENY)와 **패턴 매칭 규칙**의 동작 원리를 배웁니다
- Step 6에서 구현한 **Hooks와 권한 시스템이 통합**되는 방식을 확인합니다

> **왜 권한 시스템이 필요한가?**
> 에이전트가 `rm -rf /`를 실행하거나 민감한 파일을 수정하려 할 때, 무조건 실행하면 치명적입니다.
> Claude Code는 **"fail-closed"** 원칙을 따릅니다 — 명시적으로 허용하지 않은 것은 기본적으로 차단합니다.

---

## Claude Code 소스 분석

### 7-1. ToolPermissionContext — 권한 결정에 필요한 정보 (src/Tool.ts:123-148)

Claude Code에서 도구 실행 전에 권한을 검사할 때, 다음 컨텍스트가 전달됩니다:

```typescript
// src/Tool.ts:123-148 — ToolPermissionContext

interface ToolPermissionContext {
  tool: Tool                    // 실행하려는 도구
  input: ToolInput              // 도구에 전달할 인자
  mode: PermissionMode          // AUTO | ASK | DENY
  allowRules: PermissionRule[]  // 허용 규칙 목록
  denyRules: PermissionRule[]   // 거부 규칙 목록
  askRules: PermissionRule[]    // 사용자 확인 규칙 목록
}
```

**핵심 설계:** 권한 결정에 필요한 모든 정보가 하나의 객체에 담깁니다. 이 객체가 다단계 파이프라인을 통과하면서 최종 결정이 내려집니다.

### 7-2. Permission Decision Pipeline (src/hooks/toolPermission/PermissionContext.ts:96-348)

권한 검사는 **파이프라인** 형태로 진행됩니다:

```
도구 실행 요청
    │
    ▼
[Stage 1] 모드 확인 ─── AUTO → 즉시 허용
    │                    DENY → 즉시 거부
    ▼
[Stage 2] 정적 규칙 매칭 ─── deny 규칙 매칭 → 즉시 거부
    │                         allow 규칙 매칭 → 즉시 허용
    ▼
[Stage 3] Hooks 검사 ─── hook이 거부 → 즉시 거부
    │
    ▼
[Stage 4] 읽기전용 도구? ─── 예 → 기본 허용
    │
    ▼
[Stage 5] 사용자 프롬프트 ─── ASK 모드일 때 사용자에게 확인
```

**중요한 원칙:**
- **deny 규칙은 항상 최우선** — allow 규칙보다 먼저 평가됩니다
- **읽기전용 도구는 기본 허용** — `FileRead`, `Glob`, `Grep` 등은 명시적 deny가 없으면 허용
- **fail-closed** — 규칙에 매칭되지 않으면 사용자 확인으로 fall-through

### 7-3. 도구 생명주기에서 권한 검사 위치 (src/services/tools/toolExecution.ts)

```typescript
// src/services/tools/toolExecution.ts — 도구 실행 흐름

async function executeToolCall(toolCall, context) {
  // 1. 도구 찾기
  const tool = registry.get(toolCall.name)

  // 2. ★ 권한 검사 — 여기서 다단계 파이프라인 실행
  const permission = await checkPermissions({
    tool, input: toolCall.input,
    mode: context.permissionMode,
    allowRules: context.rules.allow,
    denyRules: context.rules.deny,
  })

  if (permission.denied) {
    return { error: permission.reason }  // 실행하지 않고 거부 사유 반환
  }

  // 3. 권한 통과 후에만 도구 실행
  const result = await tool.execute(toolCall.input)
  return result
}
```

**설계 포인트:** 권한 검사가 실패하면 도구를 아예 실행하지 않습니다. 거부 사유를 LLM에게 돌려주면, LLM은 다른 접근 방식을 시도할 수 있습니다.

In [ ]:
import sys, os
sys.path.insert(0, ".")

# mini_claude/permissions.py의 핵심 구조를 살펴봅시다

from mini_claude.permissions import (
    PermissionMode,
    PermissionAction,
    PermissionRule,
    PermissionResult,
    PermissionCheckResult,
    PermissionSystem,
    HookRegistry,
    create_default_rules,
)

# --- 1. PermissionMode: 전역 권한 모드 ---
print("=== PermissionMode ===")
for mode in PermissionMode:
    print(f"  {mode.name:6s} = {mode.value!r}")

print()

# --- 2. PermissionAction: 규칙이 취할 수 있는 액션 ---
print("=== PermissionAction ===")
for action in PermissionAction:
    print(f"  {action.name:6s} = {action.value!r}")

print()

# --- 3. PermissionRule: 패턴 매칭 기반 규칙 ---
print("=== PermissionRule 예시 ===")
rule = PermissionRule(tool_pattern="Bash", action=PermissionAction.DENY, input_pattern="*rm -rf /*")
print(f"  규칙: tool={rule.tool_pattern}, input={rule.input_pattern}, action={rule.action.value}")
print(f"  'Bash' + 'rm -rf /' 매칭? {rule.matches('Bash', 'rm -rf /')}")
print(f"  'Bash' + 'ls -la' 매칭?   {rule.matches('Bash', 'ls -la')}")
print(f"  'FileRead' 매칭?           {rule.matches('FileRead', 'test.py')}")

---

## Python으로 구현하기

### 구현 1: PermissionSystem — 다단계 파이프라인

`mini_claude/permissions.py`의 `PermissionSystem` 클래스는 Claude Code의 권한 파이프라인을 Python으로 구현합니다.
아래에서 기본 규칙 프리셋을 로드하고, 각 스테이지가 어떻게 동작하는지 확인합니다.

```python
# PermissionSystem의 check_permission() 파이프라인:
#
# Stage 1: mode 확인      → AUTO면 즉시 허용, DENY면 즉시 거부
# Stage 2: 정적 규칙 매칭  → deny/allow/ask 규칙 순서대로 매칭
# Stage 3: Hooks 검사      → 등록된 hook 콜백 실행 (AND 로직)
# Stage 4: 읽기전용 확인   → is_read_only=True면 기본 허용
# Stage 5: 사용자 프롬프트  → ASK 모드일 때 사용자에게 확인
```

In [ ]:
import asyncio

# 기본 규칙 프리셋 확인
default_rules = create_default_rules()
print("=== 기본 권한 규칙 (create_default_rules) ===\n")
for i, rule in enumerate(default_rules):
    input_info = f", input={rule.input_pattern!r}" if rule.input_pattern else ""
    print(f"  [{i}] tool={rule.tool_pattern!r}{input_info} → {rule.action.value}")

print(f"\n총 {len(default_rules)}개 규칙")
print("\n주목: deny 규칙이 allow 규칙 사이에 섞여 있지만,")
print("규칙은 순서대로 평가되며 첫 번째 매칭이 적용됩니다.")

### 구현 2: Hooks와 권한 시스템의 통합

Step 6에서 구현한 `HookRegistry`가 권한 파이프라인의 Stage 3에 통합됩니다.
Hook은 외부 프로세스(또는 Python 콜백)가 도구 실행을 허용/거부할 수 있게 합니다.

```python
# permissions.py의 HookRegistry — hooks.py의 HookRegistry와 별도
# (권한 전용 hook으로, tool_permission 이벤트만 처리)
#
# 실행 로직:
#   모든 hook이 허용 → 최종 허용 (AND 로직)
#   하나라도 거부 → 즉시 거부
#   hook 실행 오류 → 거부 (fail-closed)
```

---

## 실습: 다단계 권한 검사 데모

다양한 도구 호출 시나리오를 통해 권한 파이프라인의 각 스테이지가 어떻게 동작하는지 확인합니다.

**시나리오:**
1. `rm -rf /` — deny 규칙에 의해 즉시 차단
2. `FileRead` — 읽기 도구는 allow 규칙에 의해 자동 허용
3. `git status` — 안전한 명령은 allow 규칙에 의해 허용
4. `unknown_tool` — 매칭 규칙 없음 → 사용자 프롬프트로 fall-through
5. Hook이 거부하는 경우

In [ ]:
async def run_permission_demos():
    """다양한 시나리오에서 권한 시스템의 동작을 확인합니다."""

    # --- 기본 설정: ASK 모드 + 기본 규칙 ---
    perm = PermissionSystem(
        mode=PermissionMode.ASK,
        rules=create_default_rules(),
    )

    print("=" * 65)
    print("시나리오 1: rm -rf / — deny 규칙에 의해 차단")
    print("=" * 65)
    result = await perm.check_permission(
        "Bash", {"command": "rm -rf /important_data"}
    )
    print(f"  결과: {result.result.value}")
    print(f"  이유: {result.reason}")
    print(f"  매칭 규칙: tool={result.matched_rule.tool_pattern}, "
          f"input={result.matched_rule.input_pattern}" if result.matched_rule else "")

    print()
    print("=" * 65)
    print("시나리오 2: FileRead — allow 규칙에 의해 즉시 허용")
    print("=" * 65)
    result = await perm.check_permission(
        "FileRead", {"file_path": "/etc/passwd"}
    )
    print(f"  결과: {result.result.value}")
    print(f"  이유: {result.reason}")

    print()
    print("=" * 65)
    print("시나리오 3: git status — 안전한 명령은 허용")
    print("=" * 65)
    result = await perm.check_permission(
        "Bash", {"command": "git status"}
    )
    print(f"  결과: {result.result.value}")
    print(f"  이유: {result.reason}")

    print()
    print("=" * 65)
    print("시나리오 4: 미지의 도구 — 사용자 프롬프트로 fall-through")
    print("=" * 65)
    result = await perm.check_permission(
        "DeployToProd", {"target": "production"}
    )
    print(f"  결과: {result.result.value}")
    print(f"  이유: {result.reason}")
    print("  (기본 user_prompt는 항상 거부 — 비대화형 환경)")

    print()
    print("=" * 65)
    print("시나리오 5: Hook이 도구 실행을 거부")
    print("=" * 65)

    # Hook 등록: 근무 시간 외 도구 사용 차단 (데모용)
    async def work_hours_hook(tool_name, tool_input):
        # 데모: 항상 "근무 시간 외"로 거부
        return False, "근무 시간(09:00-18:00) 외에는 쓰기 도구를 사용할 수 없습니다"

    perm_with_hook = PermissionSystem(
        mode=PermissionMode.ASK,
        rules=create_default_rules(),
    )
    perm_with_hook.hooks.register(work_hours_hook)

    result = await perm_with_hook.check_permission(
        "Bash", {"command": "npm install express"}
    )
    print(f"  결과: {result.result.value}")
    print(f"  이유: {result.reason}")

    print()
    print("=" * 65)
    print("시나리오 6: AUTO 모드 — 모든 도구 자동 허용 (위험!)")
    print("=" * 65)
    auto_perm = PermissionSystem(mode=PermissionMode.AUTO)
    result = await auto_perm.check_permission(
        "Bash", {"command": "rm -rf /"}
    )
    print(f"  결과: {result.result.value}")
    print(f"  이유: {result.reason}")
    print("  (--dangerously-skip-permissions 플래그에 해당)")

await run_permission_demos()

In [ ]:
# 커스텀 사용자 프롬프트로 대화형 권한 확인 시뮬레이션

async def simulated_user_prompt(tool_name, tool_input, description):
    """사용자가 허용한다고 가정하는 시뮬레이션"""
    print(f"    [사용자에게 확인] {description}")
    print(f"    도구: {tool_name}, 입력: {tool_input}")
    print(f"    → 사용자가 '허용'을 선택했다고 가정합니다")
    return True  # 허용

async def demo_interactive_permission():
    perm = PermissionSystem(
        mode=PermissionMode.ASK,
        rules=create_default_rules(),
        user_prompt=simulated_user_prompt,
    )

    print("=== 대화형 권한 확인 시뮬레이션 ===\n")

    # FileWrite는 ASK 규칙 → 사용자 확인 필요
    print("--- FileWrite 도구 (ASK 규칙 매칭) ---")
    result = await perm.check_permission(
        "FileWrite", {"file_path": "/tmp/output.txt", "content": "hello"}
    )
    print(f"  최종 결과: {result.result.value}")
    print(f"  이유: {result.reason}")

    print()

    # Bash + pip install도 ASK 규칙으로 fall-through
    print("--- Bash + 'pip install' (ASK 규칙 매칭) ---")
    result = await perm.check_permission(
        "Bash", {"command": "pip install requests"}
    )
    print(f"  최종 결과: {result.result.value}")
    print(f"  이유: {result.reason}")

await demo_interactive_permission()

---

## 핵심 설계 원칙 정리

### 1. Fail-Closed (기본 거부)
명시적으로 허용하지 않은 도구는 기본적으로 사용자 확인 또는 거부됩니다. 보안 시스템에서 가장 중요한 원칙입니다. Claude Code는 `rm -rf /` 같은 위험한 명령이 규칙에 없더라도, 사용자에게 반드시 확인을 요청합니다.

### 2. 다단계 파이프라인 (Defense in Depth)
모드 확인 → 규칙 매칭 → Hooks → 읽기전용 확인 → 사용자 프롬프트. 각 단계가 독립적으로 거부할 수 있어, 한 단계가 실패해도 다른 단계가 보호합니다.

### 3. 규칙 기반 패턴 매칭
`settings.json`에서 glob 패턴으로 규칙을 정의합니다. `"Bash"` + `"git *"` → allow처럼 도구 이름과 입력 패턴을 조합하여 세밀한 제어가 가능합니다. 순서가 중요합니다 -- 첫 번째 매칭 규칙이 적용됩니다.

### 4. Hooks를 통한 확장성
권한 시스템에 외부 로직을 플러그인할 수 있습니다. 근무 시간 제한, 감사 로그, 승인 워크플로우 등을 hook으로 구현할 수 있으며, 이는 Step 6에서 배운 Hooks 시스템의 실전 활용입니다.

---

## 다음 Step 미리보기

권한 시스템으로 도구 실행을 안전하게 제어할 수 있게 되었습니다. 하지만 대화가 길어지면 또 다른 문제가 생깁니다 -- **컨텍스트 윈도우 초과**와 **API 에러**.

**Step 8: 컨텍스트 압축 + 에러 복구**에서는:
- **auto_compact** (LLM 요약), **micro_compact** (오래된 결과 제거), **reactive compact** (에러 후 복구)
- 불변 State에 **transition과 recovery 카운트**를 기록하는 패턴
- **계층적 복구:** collapse → compact → error surface

이 세 가지 전략으로 에이전트를 견고하게 만드는 방법을 배웁니다.